# Data Preparation: Temporal Train/Val/Test Split

This notebook mirrors the **Data Preparation** guide at [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/).

Demonstrate time-aware dataset splitting — a critical technique for time-series and event-driven datasets where random splits would leak future information into training.

**What we'll cover:**
1. Why temporal splits prevent data leakage
2. Generate a time-stamped dataset
3. Define the temporal split function
4. Apply and inspect the splits

In [ ]:
import pandas as pd
import numpy as np   # numpy imported twice in original — one import is sufficient


## Step 1: Why temporal splits prevent data leakage

With a random 80/20 split, a model trained in January can see observations from March — which are "in the future" relative to training time. The model learns patterns from data it wouldn't have in production.

A **temporal split** enforces strict chronological ordering:
- **Train**: all observations before `val_start`
- **Val**: observations between `val_start` and `test_start`
- **Test**: all observations from `test_start` onwards

This mirrors production: the model trains on history and is evaluated on future data it never saw.

## Step 2: Generate a time-stamped dataset

In [ ]:
np.random.seed(42)
n = 1470

# pd.date_range creates a DatetimeIndex with evenly spaced timestamps.
# freq='12h' = one observation every 12 hours.
# 1,470 observations × 12h = ~735 days ≈ 2.4 years of data.
# This simulates an event log where each row records an employee state at a specific time.
dates = pd.date_range('2023-01-01', periods=n, freq='12h')

user_features = pd.DataFrame({
    'label_date':     dates,           # Timestamp column — the split key
    'Age':            np.random.randint(22, 60, n),
    'JobLevel':       np.random.randint(1, 5, n),
    'MonthlyIncome':  np.random.randint(1500, 20000, n),
    'YearsAtCompany': np.random.randint(0, 20, n),
    # OverTime: 72% No, 28% Yes — reflects real-world distribution
    'OverTime':       np.random.choice([0, 1], n, p=[0.72, 0.28]),
    # Attrition: 84% No, 16% Yes — the imbalanced target class
    'Attrition':      np.random.choice([0, 1], n, p=[0.84, 0.16]),
})

print(f'Dataset: {len(user_features)} rows')
print(f'Date range: {user_features.label_date.min().date()} → {user_features.label_date.max().date()}')
user_features.head()


The dataset spans roughly 2.4 years (1,470 observations × 12h each). Each row represents one employee observation at a specific point in time. We'll split at two cut-off dates to create non-overlapping train / val / test sets.

## Step 3: Define the temporal split function

In [ ]:
def temporal_split(df, date_col, val_start, test_start):
    """
    Split df into train/val/test using strict chronological cut-off dates.

    Args:
        df:         DataFrame containing a datetime column
        date_col:   Name of the datetime column used as the split key
        val_start:  ISO date string — all rows on or after this date go to val (or test)
        test_start: ISO date string — all rows on or after this date go to test

    Returns:
        train_df, val_df, test_df — non-overlapping, chronologically ordered splits
    """
    # pd.Timestamp() converts an ISO string like '2024-06-01' to a Timestamp object
    # that can be compared directly with datetime column values using < and >=.
    val_start  = pd.Timestamp(val_start)
    test_start = pd.Timestamp(test_start)

    # Strict boundary conditions ensure no overlap between splits:
    # train: [start, val_start)  — strictly before the validation cut-off
    # val:   [val_start, test_start) — at or after val_start, before test_start
    # test:  [test_start, end]   — at or after the test cut-off
    train_df = df[df[date_col] <  val_start].copy()
    val_df   = df[(df[date_col] >= val_start) & (df[date_col] < test_start)].copy()
    test_df  = df[df[date_col] >= test_start].copy()

    # Print the date range of each split to visually confirm there are no gaps or overlaps.
    for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        lo = split[date_col].min().date() if len(split) else 'n/a'
        hi = split[date_col].max().date() if len(split) else 'n/a'
        print(f'{name:5s}: {len(split):5d} rows  {lo} → {hi}')

    return train_df, val_df, test_df


## Step 4: Apply and inspect the splits

In [ ]:
train_df, val_df, test_df = temporal_split(
    user_features,
    date_col='label_date',
    val_start='2024-06-01',
    test_start='2025-01-01',
)

print()
print(f'Attrition rate — Train: {train_df["Attrition"].mean():.1%}  '
      f'Val: {val_df["Attrition"].mean():.1%}  '
      f'Test: {test_df["Attrition"].mean():.1%}')

Each split covers a distinct time window with no overlap. Note that the attrition rate may differ slightly between splits — this is expected and mirrors real-world distribution shift over time. In production, you would retrain periodically to adapt to this shift.

## Next Steps

- Continue to feature stores: `02-pipeline/data-preparation/feature_store.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/)